In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from pathlib import Path
import sys

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SRC_PATH = PROJECT_PATH / "src"

PROCESSED_PATH = PROJECT_PATH / "datasets" / "processed"

sys.path.append(str(SRC_PATH))

In [4]:
data = np.load(
    PROCESSED_PATH / "uah_dataset.npz"
)

X = data["X"]
y = data["y"]
groups = data["groups"]

In [5]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

print("X :", X.shape)
print("y :", y.shape)
print("groups :", groups.shape)

print()

print("Class Distribution")

classes, counts = np.unique(y, return_counts=True)

for c, n in zip(classes, counts):
    print(f"Class {c}: {n}")

Dataset Information
X : (30676, 120, 13)
y : (30676,)
groups : (30676,)

Class Distribution
Class 0: 12991
Class 1: 9846
Class 2: 7839


# ==========================================================
# 3. GROUP K-FOLD
# ==========================================================

In [6]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

print(gkf)

GroupKFold(n_splits=5, random_state=None, shuffle=False)


In [7]:
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print("=" * 60)
    print(f"Fold {fold + 1}")
    print("=" * 60)

    print("Train :", X[train_idx].shape)
    print("Validation :", X[val_idx].shape)

    print()

Fold 1
Train : (24572, 120, 13)
Validation : (6104, 120, 13)

Fold 2
Train : (24480, 120, 13)
Validation : (6196, 120, 13)

Fold 3
Train : (24633, 120, 13)
Validation : (6043, 120, 13)

Fold 4
Train : (24544, 120, 13)
Validation : (6132, 120, 13)

Fold 5
Train : (24475, 120, 13)
Validation : (6201, 120, 13)



# ==========================================================
# 4. CREATE DATALOADERS
# ==========================================================

In [8]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import create_dataloader

In [9]:
train_idx, val_idx = next(
    gkf.split(X, y, groups)
)

X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]

print("Train :", X_train.shape)
print("Validation :", X_val.shape)

Train : (24572, 120, 13)
Validation : (6104, 120, 13)


In [10]:
train_loader = create_dataloader(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True,
)

val_loader = create_dataloader(
    X_val,
    y_val,
    batch_size=32,
    shuffle=False,
)

print("Train Loader :", len(train_loader))
print("Val Loader   :", len(val_loader))

Train Loader : 768
Val Loader   : 191


In [14]:
# ==========================================================
# CONFIGURATION
# ==========================================================

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 20
PATIENCE = 3

# ==========================================================
# 5. LSTM MODEL
# ==========================================================

In [11]:
import importlib
import lstm_model

importlib.reload(lstm_model)

from lstm_model import LSTMClassifier

In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [13]:
model = LSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

print(model)

LSTMClassifier(
  (lstm): LSTM(13, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)


# ==========================================================
# 6. BASELINE TRAINING
# ==========================================================

In [15]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import fit

In [16]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9143 | Train Acc: 0.5291 | Val Loss: 1.0787 | Val Acc: 0.5308
Epoch 2/20 | Train Loss: 0.7845 | Train Acc: 0.6183 | Val Loss: 1.2941 | Val Acc: 0.3121
Epoch 3/20 | Train Loss: 0.7197 | Train Acc: 0.6639 | Val Loss: 1.1329 | Val Acc: 0.4456
Epoch 4/20 | Train Loss: 0.6635 | Train Acc: 0.7035 | Val Loss: 1.0665 | Val Acc: 0.5125

Early stopping at epoch 4
Best Validation Accuracy : 0.5308


# ==========================================================
# 7. EVALUATION
# ==========================================================

In [17]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import evaluate_model

In [18]:
labels, predictions = evaluate_model(
    model,
    val_loader,
    device,
)

In [19]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ]
    )
)

              precision    recall  f1-score   support

      NORMAL       0.21      0.98      0.34       688
  AGGRESSIVE       0.73      0.39      0.51      2199
      DROWSY       0.96      0.49      0.65      3217

    accuracy                           0.51      6104
   macro avg       0.63      0.62      0.50      6104
weighted avg       0.79      0.51      0.57      6104



In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, predictions)

print(cm)

[[ 672    0   16]
 [1277  864   58]
 [1299  326 1592]]
